# Creating Custom Aggregates with DynamicEventAggregator


## 📋 Prerequisites

- Python installed
- Required libraries: `numpy`, `pandas`, `skillcorner`

```bash
pip install numpy pandas skillcorner
```


## 📋 Step 1: Setup & Prerequisites


In [1]:
import numpy as np
import pandas as pd
from skillcorner.client import SkillcornerClient

from opendata.features.DynamicEventsAggregator import DynamicEventAggregator

# sys.path.append("../../../")


## 📥 Step 2: Load Dynamic Events Data


In [2]:
import os

match_id = 1886347

# Instantiate the SkillCorner client
username = os.environ.get("SKILLCORNER_USERNAME")
password = os.environ.get("SKILLCORNER_PASSWORD")
client = SkillcornerClient(username=username, password=password)

# Load dynamic event data from SkillCorner API
# events_df = pd.read_csv(BytesIO(client.get_dynamic_events(MATCH_ID)))

# Load dynamic event data from Open Source
events_df = pd.read_csv(
    f"../../../data/matches/{match_id}/{match_id}_dynamic_events.csv",
)
print(len(events_df))


5079


In [3]:
events_df.shape


(5079, 294)

In [6]:
[col_name for col_name in events_df.columns if "player_id" in col_name]


['player_id']

## 📊 Step 3: Generate Standard Aggregates


In [ ]:
# Initialize the aggregator
events_aggregator = DynamicEventAggregator(df=events_df)

# Off-ball runs
off_ball_runs = events_aggregator.generate_aggregates(
    group_by=["player_id", "player_name"],
    aggregate_type="off_ball_runs",
)

# Line-breaking passes
line_breaking_passes = events_aggregator.generate_aggregates(
    group_by=["player_in_possession_id", "player_in_possession_name"],
    aggregate_type="line_breaking_passes",
)

# Defensive engagements
defensive_engagements = events_aggregator.generate_aggregates(
    group_by=["player_id", "player_name"],
    aggregate_type="on_ball_engagements",
)

# Pressing
pressing = events_aggregator.generate_aggregates(
    group_by=["player_id", "player_name"],
    aggregate_type="pressing_engagements",
)

off_ball_runs.head()



This is off_ball_runs group
	rows in `off_ball_runs` subset = 599
	rows in `off_ball_runs_in_finish` subset = 183
	rows in `off_ball_runs_in_create` subset = 255
	rows in `off_ball_runs_in_build_up` subset = 69
	rows in `off_ball_runs_in_transition` subset = 25
	rows in `cross_receiver` subset = 61
	rows in `cross_receiver_in_finish` subset = 46
	rows in `runs_in_behind` subset = 42
	rows in `runs_in_behind_in_finish` subset = 10
	rows in `runs_in_behind_in_create` subset = 23
	rows in `runs_in_behind_in_transition` subset = 2
	rows in `runs_ahead_of_the_ball` subset = 147
	rows in `runs_ahead_of_the_ball_in_finish` subset = 37
	rows in `runs_ahead_of_the_ball_in_create` subset = 69
	rows in `runs_ahead_of_the_ball_in_transition` subset = 7
	rows in `support_runs` subset = 74
	rows in `support_runs_in_finish` subset = 30
	rows in `support_runs_in_create` subset = 32
	rows in `support_runs_in_transition` subset = 3
	rows in `overlap_runs` subset = 15
	rows in `overlap_runs_in_finish` s

,player_id,player_name,count_off_ball_runs,count_targeted_off_ball_runs,count_received_off_ball_runs,xthreat_off_ball_runs,xthreat_targeted_off_ball_runs,xthreat_received_off_ball_runs,xpass_completion_off_ball_runs,xpass_completion_targeted_off_ball_runs,...,count_dangerous_received_dropping_off_runs_in_create,count_difficult_dropping_off_runs_in_create,count_difficult_targeted_dropping_off_runs_in_create,count_difficult_received_dropping_off_runs_in_create,avg_speed_avg_dropping_off_runs_in_create,count_hsr_dropping_off_runs_in_create,count_sprint_dropping_off_runs_in_create,avg_distance_covered_dropping_off_runs_in_create,count_center_channel_dropping_off_runs_in_create,count_wide_channel_dropping_off_runs_in_create
0,14736,L. Verstraete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23418,F. Gallegos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,33697,N. Pijnaker,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,38673,G. May,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,43829,N. Moreno,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Note there are now 821 columns !


## 🛠️ Step 4: Create Custom Aggregates


In [ ]:
contexts = {
    "custom": {
        "all_possessions": (
            (events_df["event_type"] == "player_possession")
            & (events_df["team_in_possession_phase_type"] == "finish")
            & (events_df["separation_start"] >= 5)
        )
    }
}


### Define Custom Metrics


In [ ]:
metric = {
    "custom": {
        "count": lambda x: len(x),
        "avg_duration": lambda x: x["duration"].mean(),
        "avg_distance_covered": lambda x: x["distance_covered"].mean(),
    }
}


### Generate Custom Aggregates


In [ ]:
# Initialize aggregator with custom contexts and metrics
custom_events_aggregator = DynamicEventAggregator(
    df=events_df, custom_context_groups=contexts, custom_metric_groups=metric
)

# Generate custom aggregates
custom_aggregates = custom_events_aggregator.generate_aggregates(
    group_by=["player_id", "player_name"], aggregate_type="custom"
)

# Display the results
custom_aggregates.head()


/Users/nano/PycharmProjects/opendata/resources/Tutorials/Aggregating Dynamic Events/DynamicEventsAggregator.py:948: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  context_df.groupby(group_by + ['context'])


,player_id,player_name,count_all_possessions,avg_duration_all_possessions,avg_distance_covered_all_possessions
0,14736,L. Verstraete,9.0,1.966667,4.686667
1,23418,F. Gallegos,10.0,1.020000,2.167000
2,33697,N. Pijnaker,12.0,1.066667,3.006667
3,38673,G. May,2.0,2.000000,10.705000
4,43829,N. Moreno,2.0,2.800000,10.400000
